## Data Preprocessing

In [ ]:
from datasets import load_dataset
from PIL import Image
from transformers import BertTokenizer
import matplotlib.pyplot as plt

# Load the dataset (select only the first sample for this example)
flickr_dataset = load_dataset("nlphuji/flickr30k", split="test").select(range(1))

# Extract the first image and its captions
try:
    first_image = flickr_dataset[0]["image"]
    first_image_captions = flickr_dataset[0]["caption"]
except Exception as e:
    print(f"Error accessing data: {e}")
    exit()

# Print the captions
print("Original Captions:")
for cap in first_image_captions:
    print("- ", cap)

# Display the image
plt.imshow(first_image)
plt.axis("off")
plt.title("First Image from Dataset")
plt.show()

# --- Now, let's simulate what the ImageCaptionDataset and DataLoader would do ---

# 1. Initialize Tokenizer (same as in your main code)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# 2. Build a simplified Vocabulary (for demonstration purposes)
# In your actual code, you build a more sophisticated vocabulary, but here we'll just
# manually create a small one to illustrate the process.
class SimpleVocab:
    def __init__(self, tokenizer):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0
        self.tokenizer = tokenizer

        # Add special tokens
        for special_token in [tokenizer.pad_token, tokenizer.unk_token, tokenizer.cls_token, tokenizer.sep_token]:
            if special_token is not None:
                self.add_word(special_token)

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if word not in self.word2idx:
            return self.word2idx.get(self.tokenizer.unk_token, 0)
        return self.word2idx[word]
    
    def __len__(self):
        return len(self.word2idx)

# Create an instance of the simple vocabulary
vocab = SimpleVocab(tokenizer)

# Manually add some words from the first image's captions to the vocabulary
for cap in first_image_captions:
    tokens = tokenizer.tokenize(cap.lower())
    for token in tokens:
        vocab.add_word(token)


# 3. Simulate ImageCaptionDataset's __getitem__
def simulate_getitem(image, caption_list, tokenizer, vocab):
    # Tokenize and vectorize caption
    combined_caption = " ".join(caption_list)
    tokens = tokenizer.tokenize(str(combined_caption).lower())
    caption_vec = [vocab(tokenizer.cls_token)]
    caption_vec.extend([vocab(token) for token in tokens])
    caption_vec.append(vocab(tokenizer.sep_token))

    print("\nSimulated Tokenized Caption (using SimpleVocab):")
    print(caption_vec)

    print("\nCorresponding Words (using SimpleVocab):")
    print([vocab.idx2word.get(idx, tokenizer.unk_token) for idx in caption_vec])

    return image, caption_vec

# Call the simulation function
simulated_image, simulated_caption_vec = simulate_getitem(
    first_image, first_image_captions, tokenizer, vocab
)

# 4. Display the image again (for verification)
plt.imshow(simulated_image)
plt.axis("off")
plt.title("Image (after simulated __getitem__)")
plt.show()

print("\nIf the printed captions, tokenized captions, corresponding words, and the two images match,")
print("then it's a good indication that your ImageCaptionDataset and DataLoader are likely parsing")
print("the images and captions correctly.")